# 02 · Analyze — every hypothesis, every figure, the verdict memo

**No GPU, runs in minutes, and safe to run early.** Attach every finished worker session
(*Add Input → Your Work*, one per task) plus the preparation session. Each section reports what
it is missing rather than quietly analysing a partial study, so this is worth running as soon
as Tier A lands and again after Tier B.

Its own output is tables and figures — a few tens of MB. The 20 GB cap is not in play here;
the guard runs anyway, because a guard you only run when you are worried is not a guard.

Order follows the hypotheses, not what came out best:

1. the paired per-case table — Dice, NSD at 2 mm, HD95, detection, false positives (deliv. 1)
2. **H1** volume, **H2** occlusion + the CNN effective receptive field, **H2b** identity
   control, **H3** source shift against the boundary-tolerance floor
3. the contrast sensitivity analysis and its pre-registered exploratory/confirmatory gate
4. the negative control, seed-versus-fold variance, the rule-selected failure gallery
5. six figures and the verdict memo

In [ ]:
# === PDAC study bootstrap =============================================================
# Identical in all three notebooks. Paths, environment, repo, disk budget, and the
# cross-session state helpers.
import os, sys, subprocess, shutil, json, tarfile, textwrap, time
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
REPO_URL  = "https://github.com/spraldev/pdac-inductive-bias.git"

# --- the two hard limits, in one place -------------------------------------------------
# Kaggle kills a session at 12 h and refuses to save an output larger than 20 GB. Both are
# silent failures if you meet them by accident, so both are budgeted with headroom and
# checked rather than hoped for.
SESSION_HOURS   = float(os.environ.get("PDAC_SESSION_HOURS", 11.0))   # of a 12 h cap
OUTPUT_LIMIT_GB = float(os.environ.get("PDAC_OUTPUT_LIMIT_GB", 17.0)) # of a 20 GB cap
SESSION_T0 = time.time()

# --- paths ------------------------------------------------------------------------------
# WORK persists as the notebook's saved output and is what the 20 GB cap applies to: only
# small, precious, or genuinely needed-downstream things go there. SCRATCH is much larger and
# is wiped with the session, so everything regenerable lives there — raw images, nnU-Net
# preprocessed data, occluded volumes.
if ON_KAGGLE:
    WORK    = Path("/kaggle/working")
    SCRATCH = Path("/kaggle/temp/pdac"); SCRATCH.mkdir(parents=True, exist_ok=True)
    REPO    = WORK / "pdac-research"
else:
    WORK    = Path(os.environ.get("PDAC_WORK", Path.cwd() / "pdac_work"))
    SCRATCH = Path(os.environ.get("PDAC_SCRATCH", WORK / "scratch"))
    REPO    = Path(os.environ.get("PDAC_REPO", Path.cwd()))
    WORK.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path(os.environ.get("PDAC_DATA", SCRATCH / "data"))
RESULTS   = WORK / "results";  RESULTS.mkdir(parents=True, exist_ok=True)
STATE     = WORK / "state";    STATE.mkdir(parents=True, exist_ok=True)
PREDS     = WORK / "preds";    PREDS.mkdir(parents=True, exist_ok=True)

# --- repo ---------------------------------------------------------------------------------
def _find_attached(*required):
    """First attached input containing all of the given relative paths."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for p in sorted(root.glob("*")):
        for cand in [p] + sorted(x for x in p.glob("*") if x.is_dir()):
            if all((cand / r).exists() for r in required):
                return cand
    return None

if ON_KAGGLE and not (REPO / "config" / "analysis_config.yaml").exists():
    src = _find_attached("config/analysis_config.yaml")
    if src is not None:
        print(f"Using repo attached as a dataset: {src}")
        shutil.copytree(src, REPO, dirs_exist_ok=True)
    else:
        print("Cloning the repo (Internet must be on in notebook settings) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts" / "analysis"))
sys.path.insert(0, str(REPO / "scripts" / "training"))
os.environ["PDAC_REPO"] = str(REPO)

# These notebooks are generated FROM the repo but, on Kaggle, run AGAINST a clone of it — so
# an uncommitted or unpushed change is invisible here no matter how current the notebook is.
# When the clone predates the notebook the symptom lands far from the cause: an old test
# asserting old counts, a script missing a flag this notebook passes. Checking the contract
# up front turns that into one clear message.
_REQUIRED = ["config/analysis_config.yaml", "scripts/training/tasks.py",
             "scripts/analysis/build_per_case_table.py", "scripts/analysis/make_figures.py",
             "src/trainers/budget_trainers.py", "scripts/kaggle/pack_for_kaggle.py"]
_missing = [r for r in _REQUIRED if not (REPO / r).exists()]
if _missing:
    raise RuntimeError(
        "The repo this notebook is running against is older than the notebook itself.\n"
        f"  missing: {_missing}\n"
        f"  repo:    {REPO}\n"
        "These notebooks clone " + REPO_URL + ", so local commits only reach Kaggle "
        "once they are PUSHED. "
        "Either push, or upload the repo as a Kaggle Dataset and attach it "
        "(the bootstrap prefers an attached input containing config/analysis_config.yaml).")

# --- nnU-Net environment --------------------------------------------------------------------
# raw and preprocessed are regenerable and enormous -> SCRATCH.
# results holds checkpoints, which are neither -> WORK, under the budget guard below.
os.environ["nnUNet_raw"]          = str(SCRATCH / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(SCRATCH / "nnUNet_preprocessed")
os.environ["nnUNet_results"]      = str(WORK / "nnUNet_results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

# --- shell / install helpers -------------------------------------------------------------------
def sh(cmd, cwd=None, check=True):
    """Run a shell command from the repo root, streaming output into the notebook."""
    cwd = str(cwd or REPO)
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

def pip_install(pkgs, quiet=True):
    sh(f"{sys.executable} -m pip install {'-q ' if quiet else ''}--no-warn-script-location {pkgs}")

def gpu_info():
    try:
        import torch
    except ImportError:
        print("torch not installed yet"); return None
    if not torch.cuda.is_available():
        print("No CUDA device. Turn on a GPU accelerator in notebook settings."); return None
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  ({gb:.1f} GB)")
    return {"name": name, "vram_gb": round(gb, 1)}

# --- the 20 GB guard ------------------------------------------------------------------------------
def dir_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e9

def disk_report(detail=True):
    """What the session is holding, split by which limit it counts against."""
    out = dir_gb(WORK)
    free_scratch = shutil.disk_usage(SCRATCH).free / 1e9
    print(f"OUTPUT (counts against the {OUTPUT_LIMIT_GB:.0f}/20 GB cap): {out:.2f} GB")
    if detail:
        for sub in sorted(p for p in WORK.iterdir() if p.is_dir()):
            g = dir_gb(sub)
            if g > 0.01:
                print(f"    {g:7.2f} GB  {sub.name}/")
    print(f"SCRATCH (wiped with the session, not capped): {dir_gb(SCRATCH):.2f} GB used, "
          f"{free_scratch:.0f} GB free")
    return out

def enforce_output_budget(limit_gb=None, where=""):
    """Get the output back under budget, and only then complain if it cannot be done.

    Raising alone would not help: Kaggle refuses the save regardless of what the notebook
    thinks, so an over-budget session loses its GPU hours either way. This frees space in
    increasing order of regret and re-measures after each step, so the common case (a
    checkpoint the study never evaluates) is handled silently and only a genuine overrun
    reaches the user.
    """
    limit = OUTPUT_LIMIT_GB if limit_gb is None else limit_gb
    used = dir_gb(WORK)
    if used <= limit:
        print(f"output {used:.2f} / {limit:.0f} GB{' at ' + where if where else ''}  OK")
        return used

    print(f"output {used:.2f} GB is over the {limit:.0f} GB budget — freeing space")

    # 1. Checkpoints this study never reads. Zero regret.
    prune_checkpoints()
    used = dir_gb(WORK)

    # 2. Softmax dumps and validation scratch. Nothing here reads them either; they only
    #    appear if a training command was run with --npz, which this repo no longer does.
    if used > limit:
        for pattern in ("*.npz", "*.pkl"):
            for f in Path(os.environ["nnUNet_results"]).rglob(pattern):
                print(f"  removed {f.name}"); f.unlink()
        used = dir_gb(WORK)

    # 3. Resume checkpoints for runs that finished. Costs the ability to resume a run that
    #    has nothing left to resume.
    if used > limit:
        prune_checkpoints(keep_latest_if_unfinished=False)
        used = dir_gb(WORK)

    if used > limit:
        disk_report()
        raise RuntimeError(
            f"Output is still {used:.1f} GB after pruning, over the {limit:.0f} GB budget"
            f"{' at ' + where if where else ''}. Kaggle will refuse to save this session. "
            "Drop this task's checkpoint (DROP_CHECKPOINT = True) if its predictions are "
            "already written — every analysis except the receptive-field measurement reads "
            "predictions, not weights.")
    print(f"output now {used:.2f} / {limit:.0f} GB  OK")
    return used

def time_left_h():
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600.0

def check_time(where=""):
    left = time_left_h()
    print(f"{left:.2f} h left of the {SESSION_HOURS:.1f} h session budget"
          f"{' at ' + where if where else ''}")
    return left

def prune_checkpoints(root=None, keep_latest_if_unfinished=True):
    """Keep exactly what the study needs from each run's directory.

    nnU-Net writes checkpoint_best, checkpoint_latest and checkpoint_final. Evaluation in this
    study is on checkpoint_final only (the pre-registered schedule has no early stopping), so
    best is always removable, and latest is removable the moment final exists. Left alone,
    three checkpoints per run is three times the storage for no gain.
    """
    root = Path(root or os.environ["nnUNet_results"])
    freed = 0.0
    for fold_dir in sorted(p for p in root.rglob("fold_*") if p.is_dir()):
        final = fold_dir / "checkpoint_final.pth"
        drop = [fold_dir / "checkpoint_best.pth"]
        if final.exists() or not keep_latest_if_unfinished:
            drop.append(fold_dir / "checkpoint_latest.pth")
        for f in drop:
            if f.exists():
                freed += f.stat().st_size / 1e9
                f.unlink()
                print(f"  removed {f.relative_to(root)}")
    if freed:
        print(f"freed {freed:.2f} GB")
    return freed

# --- cross-session state --------------------------------------------------------------------------
def save_state(*rel_paths):
    """Copy repo-relative paths into WORK/state so they survive as notebook output."""
    for rel in rel_paths:
        src = REPO / rel
        if not src.exists():
            print(f"  (skip, absent) {rel}"); continue
        dst = STATE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        (shutil.copytree if src.is_dir() else shutil.copy2)(
            src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
        print(f"  saved {rel}")

def restore_state(*rel_paths, required=True):
    """Restore from WORK/state or from any attached dataset holding a state/ directory."""
    sources = [STATE]
    if Path("/kaggle/input").exists():
        sources += sorted(Path("/kaggle/input").rglob("state"))
    missing = []
    for rel in rel_paths:
        for base in sources:
            src = base / rel
            if src.exists():
                dst = REPO / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                (shutil.copytree if src.is_dir() else shutil.copy2)(
                    src, dst, **({"dirs_exist_ok": True} if src.is_dir() else {}))
                print(f"  restored {rel}  <- {base}")
                break
        else:
            missing.append(rel)
    if missing:
        msg = ("Missing state: " + ", ".join(missing) + "\n  Run the preparation notebook, "
               "then attach its output here (Add Input -> Your Work).")
        if required:
            raise FileNotFoundError(msg)
        print("  " + msg)
    return not missing

def restore_chunks(dest, manifest_name="pack_manifest.json"):
    """Unpack a chunked dataset produced by scripts/kaggle/pack_for_kaggle.py.

    Large reusable data (the preprocessed cohort) cannot travel as notebook output — that is
    what the 20 GB cap forbids — so it travels as an attached dataset in size-bounded parts.
    This finds the manifest in any attached input and extracts every part into dest.
    """
    root = Path("/kaggle/input")
    if not root.exists():
        return False
    for man_path in sorted(root.rglob(manifest_name)):
        man = json.loads(man_path.read_text())
        parts = man.get("parts", [])
        print(f"Found a {len(parts)}-part pack at {man_path.parent} "
              f"({man.get('uncompressed_bytes', 0)/1e9:.1f} GB)")
        dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
        for entry in parts:
            src = man_path.parent / entry["name"]
            if not src.exists():
                print(f"  MISSING {entry['name']} — attach every part, not just some"); continue
            with tarfile.open(src) as tar:
                tar.extractall(dest)
            print(f"  extracted {entry['name']} ({entry['n_files']} files)")
        return True
    return False

print(f"ON_KAGGLE={ON_KAGGLE}\nREPO={REPO}\nWORK={WORK}\nSCRATCH={SCRATCH}\n"
      f"DATA_ROOT={DATA_ROOT}\nbudgets: {SESSION_HOURS} h session, {OUTPUT_LIMIT_GB} GB output")

In [ ]:
# Analysis environment. Kaggle already ships numpy/pandas/scipy/matplotlib; these are the rest.
pip_install("SimpleITK nibabel openpyxl pyyaml statsmodels zenodo-get "
            "'surface-distance @ git+https://github.com/google-deepmind/surface-distance.git'")
import importlib
for m in ("SimpleITK", "surface_distance", "statsmodels", "yaml", "pandas", "scipy"):
    importlib.import_module(m)
print("analysis environment OK")

In [ ]:
restore_state("splits", "config/frozen_thresholds.yaml")
import pandas as pd, yaml, itertools

# Gather every attached worker's predictions into one tree. Workers write disjoint paths
# (preds/<arm>/<fold>/, preds/occlusion/<arm>/<shell>/, preds/nih/<arm>/<condition>/), so
# merging is a copy with no conflicts to resolve.
merged = 0
if Path("/kaggle/input").exists():
    for cand in Path("/kaggle/input").rglob("preds"):
        if not cand.is_dir():
            continue
        for f in cand.rglob("*.nii.gz"):
            dst = PREDS / f.relative_to(cand)
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                shutil.copy2(f, dst); merged += 1
print(f"merged {merged} prediction files")

# Out-of-fold predictions are per fold on disk; the analysis wants one flat directory per arm.
OOF = WORK / "oof"
for arm in ("cnn", "tf", "identity_control"):
    src_root = PREDS / arm
    if not src_root.exists():
        continue
    dst = OOF / arm; dst.mkdir(parents=True, exist_ok=True)
    # "-seed" directories are replicate runs of a fold that already has an owner here; folding
    # them into the out-of-fold set would give some cases two predictions from the same arm.
    for fold_dir in sorted(d for d in src_root.glob("fold*") if "-seed" not in d.name):
        for f in fold_dir.glob("*.nii.gz"):
            if not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    print(f"{arm}: {len(list(dst.glob('*.nii.gz')))} out-of-fold predictions")

# Leave-one-source-out predictions, in the layout loso_analysis.py expects.
LOSO = WORK / "loso_preds"
for arm in ("cnn", "tf"):
    for fold_dir in sorted((PREDS / arm).glob("loso_*")) if (PREDS / arm).exists() else []:
        dst = LOSO / arm / fold_dir.name; dst.mkdir(parents=True, exist_ok=True)
        for f in fold_dir.glob("*.nii.gz"):
            if not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
        print(f"{arm}/{fold_dir.name}: {len(list(dst.glob('*.nii.gz')))}")
disk_report()

In [ ]:
# --- what is present, and what is still outstanding ----------------------------------------
# A study analysed on a partial set of runs is not the study, so this is stated up front
# rather than inferred later from a suspiciously small n.
sys.path.insert(0, str(REPO / "scripts" / "training"))
import tasks as task_mod
all_names = task_mod.all_tasks(REPO)

logs = sorted(Path("/kaggle/input").rglob("state/run_logs/*.csv")) if Path("/kaggle/input").exists() else []
logs += sorted((STATE / "run_logs").glob("*.csv"))
run_log = (pd.concat([pd.read_csv(p) for p in logs], ignore_index=True)
           if logs else pd.DataFrame(columns=["run_id", "arm", "fold", "seed", "status"]))
if len(run_log):
    run_log.to_csv(RESULTS / "run_log_merged.csv", index=False)
    print(run_log[["run_id", "arm", "fold", "seed", "status", "notes"]].to_string(index=False))

done = set()
for name in all_names:
    try:
        T = task_mod.parse(name, repo=REPO, budget_suffix="")
    except SystemExit:
        continue
    arm_tag = {"cnn": "cnn", "tf": "tf", "identity": "identity_control"}[T["arm_key"]]
    if (PREDS / arm_tag / T["pred_key"]).exists():
        done.add(name)
print(f"\n{len(done)}/{len(all_names)} tasks have predictions.")
missing = [n for n in all_names if n not in done]
if missing:
    print("outstanding:", ", ".join(missing))

In [ ]:
# --- references ------------------------------------------------------------------------------
# The reference masks come from the nnU-Net raw dataset, which is regenerable, so it is built
# into scratch rather than carried between sessions.
REFS = Path(os.environ["nnUNet_raw"]) / "Dataset501_PDAC" / "labelsTr"
IMAGES = Path(os.environ["nnUNet_raw"]) / "Dataset501_PDAC" / "imagesTr"
if not REFS.exists():
    sh(f"{sys.executable} scripts/data/convert_to_nnunet.py --data-root {DATA_ROOT} "
       f"--cohort splits/cohort.csv")
print(f"{len(list(REFS.glob('*.nii.gz')))} reference masks")

In [ ]:
# --- 1. the paired per-case table (deliverable 1) ------------------------------------------------
# Surface distances are computed once here and every hypothesis reads from this table. The
# script refuses an incomplete pairing: a paired analysis with silently dropped cases is not a
# paired analysis.
MT = RESULTS / "per_case_metrics.csv"
sh(f"{sys.executable} scripts/analysis/build_per_case_table.py "
   f"--arm cnn:{OOF / 'cnn'} --arm tf:{OOF / 'tf'} --refs {REFS} "
   f"--strata splits/strata.csv --cohort splits/cohort.csv --out {MT}", check=False)
if MT.exists():
    mt = pd.read_csv(MT)
    print(mt.groupby("arm")[["dice", "nsd2mm", "hd95", "detected", "fp_count"]]
          .mean().round(4).to_string())
    print("\n(aggregate means; nothing is claimable until the per-stratum intervals below)")

In [ ]:
# --- 2. H1 and the stratified maps ------------------------------------------------------------------
sh(f"{sys.executable} scripts/analysis/paired_analysis.py --metrics-table {MT} "
   f"--cohort splits/cohort.csv --out {RESULTS / 'h1'}", check=False)

h1p = RESULTS / "h1" / "summary.yaml"
if h1p.exists():
    h1 = yaml.safe_load(open(h1p))
    margin = yaml.safe_load(open(REPO / "config" / "analysis_config.yaml"))[
        "hypotheses"]["h1"]["equivalence_margin_dice_points"]
    print(f"H1 slope {h1['h1_slope']:+.5f}, 95% CI "
          f"[{h1['h1_ci'][0]:+.5f}, {h1['h1_ci'][1]:+.5f}] -> "
          f"{'SUPPORTED' if h1['h1_supported'] else 'NOT SUPPORTED'} (rule: CI below zero)")
    print(f"Large tertile TOST at ±{margin} points: "
          f"{'EQUIVALENT' if h1['large_tertile_equivalent'] else 'NOT SHOWN EQUIVALENT'}")
    print("\n" + pd.read_csv(RESULTS / "h1" / "stratum_heatmap.csv").round(4).to_string(index=False))

In [ ]:
# --- 3. contrast sensitivity at 5 / 10 / 15 mm rings ---------------------------------------------------
# Pre-specified: if CNR ranks move across ring widths, the contrast axis is reported as
# exploratory. That call was made in August, so applying it now is not a retreat.
sh(f"{sys.executable} scripts/analysis/contrast_sensitivity.py --metrics-table {MT} "
   f"--strata splits/strata.csv --out {RESULTS / 'contrast_sensitivity'}", check=False)

In [ ]:
# --- 4. H2: score the occlusion predictions the workers produced -----------------------------------------
# Each worker occluded and re-inferred its own held-out fold, so the shell directories here are
# already the union over folds. The scoring stage only needs them plus the baseline predictions.
OCC = SCRATCH / "occlusion_score"
have_occ = (PREDS / "occlusion").exists()
if have_occ:
    for arm in ("cnn", "tf"):
        for shell_dir in sorted((PREDS / "occlusion" / arm).glob("shell_*")):
            dst = OCC / f"pred_{arm}" / shell_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            for f in shell_dir.glob("*.nii.gz"):
                if not (dst / f.name).exists():
                    shutil.copy2(f, dst / f.name)
    sh(f"{sys.executable} scripts/analysis/occlusion_test.py --stage score --workdir {OCC} "
       f"--pred-base-cnn {OOF / 'cnn'} --pred-base-tf {OOF / 'tf'} --refs {REFS} "
       f"--cohort splits/cohort.csv --out {RESULTS / 'h2'}", check=False)
    h2p = RESULTS / "h2" / "summary.yaml"
    if h2p.exists():
        h2 = yaml.safe_load(open(h2p))
        print(f"H2 decisive shell: mean(tf loss − cnn loss) = {h2['h2_contrast_mean']:+.4f}, "
              f"CI [{h2['h2_ci'][0]:+.4f}, {h2['h2_ci'][1]:+.4f}] -> "
              f"{'SUPPORTED' if h2['h2_supported'] else 'NOT SUPPORTED'}")
else:
    print("No occlusion predictions yet — they come from the two arms' CV-fold workers.")

In [ ]:
# --- 5. the CNN effective receptive field (the H2 reference line) -------------------------------------------
# Measured from input gradients on the network that was actually trained: the theoretical
# receptive field only bounds what a voxel could see, not what it depends on.
sh(f"{sys.executable} scripts/analysis/effective_receptive_field.py --self-test", check=False)

ckpt = None
if Path("/kaggle/input").exists():
    hits = sorted(Path("/kaggle/input").rglob(
        "nnUNetTrainer*__nnUNetResEncUNet*Plans__3d_fullres/fold_0/checkpoint_final.pth"))
    ckpt = hits[0] if hits else None
if ckpt:
    # nnU-Net is a heavy install and this is the only cell in an otherwise CPU-only notebook
    # that needs it, so it goes in only when there is actually a network to rebuild.
    pip_install("'nnunetv2 @ git+https://github.com/MIC-DKFZ/nnUNet.git'")
    plans_name = ckpt.parent.parent.name.split("__")[1]
    sh(f"{sys.executable} scripts/analysis/effective_receptive_field.py "
       f"--nnunet-checkpoint {ckpt} --dataset 501 --configuration 3d_fullres "
       f"--plans {plans_name} --fold 0 --out {RESULTS / 'erf_cnn.yaml'}", check=False)
else:
    print("No CNN fold-0 checkpoint attached — attach the cnn-fold0 worker session for the "
          "receptive-field reference line on the H2 figure.")

In [ ]:
# --- 6. H3: leave-one-source-out and the boundary-tolerance floor ---------------------------------------------
# The floor needs no model output at all and is computed whether or not the LOSO runs exist;
# the share-of-loss statistic needs both.
if (LOSO / "cnn").exists() and (LOSO / "tf").exists():
    sh(f"{sys.executable} scripts/analysis/loso_analysis.py --metrics-table {MT} "
       f"--pred cnn:{LOSO / 'cnn'} --pred tf:{LOSO / 'tf'} --refs {REFS} "
       f"--cohort splits/cohort.csv --out {RESULTS / 'loso'}", check=False)
    if (RESULTS / "loso" / "by_source.csv").exists():
        print(pd.read_csv(RESULTS / "loso" / "by_source.csv").round(4).to_string(index=False))
else:
    print("No leave-one-source-out predictions yet (Tier B's cnn-loso_* / tf-loso_* tasks).")

loso_dice = RESULTS / "loso" / "loso_dice.csv"
sh(f"{sys.executable} scripts/analysis/boundary_tolerance.py --refs {REFS} "
   f"--cohort splits/cohort.csv {'--loso ' + str(loso_dice) if loso_dice.exists() else ''} "
   f"--out {RESULTS / 'h3'}", check=False)

In [ ]:
# --- 7. the inter-rater proxy ------------------------------------------------------------------------------------
# Two-sided on purpose: a list of paired cases, or a written statement that there are none. A
# synthetic boundary floor with no empirical check is a limitation to state, not to discover
# at review.
sh(f"{sys.executable} scripts/analysis/paired_delineation_check.py "
   f"--labels-root {DATA_ROOT / 'panorama' / 'panorama_labels'} --cohort splits/cohort.csv "
   f"--out {RESULTS / 'inter_rater'}", check=False)
p = RESULTS / "inter_rater" / "STATEMENT.md"
print(p.read_text() if p.exists() else "")

In [ ]:
# --- 8. H2b, the identity control -----------------------------------------------------------------------------------
if (OOF / "identity_control").exists() and any((OOF / "identity_control").glob("*.nii.gz")):
    sh(f"{sys.executable} scripts/analysis/build_per_case_table.py "
       f"--arm tf:{OOF / 'tf'} --arm identity_control:{OOF / 'identity_control'} "
       f"--refs {REFS} --strata splits/strata.csv --cohort splits/cohort.csv "
       f"--out {RESULTS / 'per_case_metrics_h2b.csv'}", check=False)
    sh(f"{sys.executable} scripts/analysis/identity_comparison.py "
       f"--metrics-table {RESULTS / 'per_case_metrics_h2b.csv'} --out {RESULTS / 'h2b'}",
       check=False)
else:
    print("No identity-control predictions yet (Tier B's identity-fold* tasks).")

In [ ]:
# --- 9. seed variance against fold variance --------------------------------------------------------------------------
# The seed replicates live under their own trainer class names, which is what stops a second
# seed on the same fold from overwriting the first.
rep = yaml.safe_load(open(REPO / "config" / "analysis_config.yaml"))["training"]["seed_replicates"]
for arm in ("cnn", "tf"):
    args = []
    for s in rep["extra_seeds"]:
        # No fallback to the default-seed directory: silently substituting the default run for
        # a missing replicate would compare a run against itself and report seed variance of
        # zero, which is worse than reporting nothing.
        d = PREDS / arm / f"fold{rep['fold']}-seed{s}"
        if d.exists():
            args += ["--seed-pred", f"{s}:{d}"]
    if not args or not MT.exists():
        print(f"{arm}: no seed replicates yet"); continue
    sh(f"{sys.executable} scripts/analysis/build_variance_tables.py --arm {arm} "
       f"--metrics-table {MT} --refs {REFS} {' '.join(args)} "
       f"--out {RESULTS / f'variance_inputs_{arm}'}", check=False)
    sh(f"{sys.executable} scripts/analysis/variance_decomposition.py --arm {arm} "
       f"--fold-dice {RESULTS / f'variance_inputs_{arm}' / 'fold_dice.csv'} "
       f"--seed-dice {RESULTS / f'variance_inputs_{arm}' / 'seed_dice.csv'} "
       f"--out {RESULTS / f'variance_{arm}.yaml'}", check=False)

In [ ]:
# --- 10. the NIH negative control (deliverable 8) --------------------------------------------------------------------
args = []
for arm, cond in itertools.product(("cnn", "tf"), ("in_domain", "source_shift")):
    d = PREDS / "nih" / arm / cond
    if d.exists() and any(d.glob("*.nii.gz")):
        args += ["--pred", f"{arm}:{cond}:{d}"]
if args:
    sh(f"{sys.executable} scripts/analysis/nih_false_positives.py {' '.join(args)} "
       f"--cohort splits/cohort.csv --out {RESULTS / 'nih_fp'}", check=False)
else:
    print("No NIH predictions yet — the fold0 and loso_* worker tasks produce them.")

In [ ]:
# --- 11. figures --------------------------------------------------------------------------------------------------------
# Each is skipped with a printed reason when its inputs are absent, so this cell is correct
# after Tier A and again after Tier B.
sh(f"{sys.executable} scripts/analysis/make_figures.py --results {RESULTS} "
   f"--out {RESULTS / 'figures'}", check=False)
from IPython.display import Image, Markdown, display
for p in sorted((RESULTS / "figures").glob("*.png")):
    print(p.name); display(Image(filename=str(p)))

In [ ]:
# --- 12. the failure gallery (deliverable 9) ------------------------------------------------------------------------------
# Selected by the frozen rule — largest |ΔDice| within each stratum cell — never by eye. The
# selection script is in the repo precisely so nobody has to be trusted about that.
if (RESULTS / "h1" / "per_case.csv").exists() and IMAGES.exists():
    sh(f"{sys.executable} scripts/analysis/failure_gallery.py "
       f"--per-case {RESULTS / 'h1' / 'per_case.csv'} --pred-cnn {OOF / 'cnn'} "
       f"--pred-tf {OOF / 'tf'} --refs {REFS} --images {IMAGES} "
       f"--out {RESULTS / 'failure_gallery'}", check=False)
    for p in sorted((RESULTS / "failure_gallery").glob("*.png")):
        print(p.name); display(Image(filename=str(p)))

In [ ]:
# --- 13. the verdict memo -----------------------------------------------------------------------------------------------------
# Rules quoted from the frozen config, numbers read from the analysis scripts' own summaries,
# and the rule applied to the number. It cannot invent a rule because it has nowhere else to
# read one from.
sh(f"{sys.executable} scripts/analysis/verdict_memo.py --results {RESULTS} "
   f"--out {RESULTS / 'VERDICT.md'}")
display(Markdown((RESULTS / "VERDICT.md").read_text()))

In [ ]:
save_state("scripts/training/run_log.csv")
disk_report()
enforce_output_budget(where="end of analysis")
print(f"\nEverything for the paper is under {RESULTS}. Copy results/ out of this notebook's "
      "output, commit it, and run: bash scripts/check_prereg_tag.sh")

## Reading this against the rules, not around them

The memo applies the frozen rules; it does not weigh or reinterpret them. A hypothesis that
reads NOT SUPPORTED is reported as a stratified negative result with its intervals, per
section 7 of the pre-registration — not re-tested under a rule invented after the fact. The
point of freezing in August was that this step is arithmetic.

Before writing any prose, run `bash scripts/check_prereg_tag.sh` and confirm PASS. If a frozen
file has changed, the numbers above are not what was pre-registered, and the difference belongs
in `preregistration/DEVIATIONS.md` and in the paper.